# Capa socioeconómica: correspondencia circuito↔radio censal y EPH Gran La Plata


1. **`circuito_radio_correspondencia.csv`**: reparte cada radio censal (2010 y 2022) entre los circuitos electorales de La Plata que intersecta, ponderado por área — prerrequisito técnico para unir cualquier variable de Censo a la tabla electoral (`src/geolocalizacion/geo.py`).
2. **`eph_gran_la_plata.csv`**: serie trimestral 2003T3-2025T4 de desocupación/informalidad/ingreso para el aglomerado Gran La Plata (`src/socioeconomia/eph_client.py`) — a nivel de aglomerado, **nunca** de circuito (ver limitación al final).

In [ ]:
import sys
from pathlib import Path

import pandas as pd


def _raiz_repo(desde: Path | None = None) -> Path:
    """Sube desde `desde` (por defecto el cwd del kernel) buscando `pytest.ini`
    como marca de la raíz del repo -- no asume `cwd == notebooks/`, así el
    notebook corre igual si Jupyter arranca desde la raíz del repo o vía
    nbconvert/CI."""
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo (buscando pytest.ini) subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

from socioeconomia.eph_client import (
    EphClient,
    TrimestreNoPublicado,
    UrlDesconocida,
    agregados_gran_la_plata,
    agregados_por_edad,
    agregados_por_sexo,
)
from geolocalizacion.geo import calcular_correspondencia, cargar_circuitos_electorales, cargar_radios_censales

SOCIOECONOMIA = REPO / "data" / "socioeconomia"

## 1. Correspondencia espacial circuito electoral ↔ radio censal

Circuitos electorales (`circuitos_electorales_la_plata.geojson`, Cámara Nacional Electoral / catálogo PBA) y radios censales (`radios_censales_2010_la_plata.geojson` / `_2022_la_plata.geojson`, cartografía armonizada CONICET 1991/2001/2010/2022) no comparten identificador — el join es espacial, no por id (ver `src/geolocalizacion/geo.py`). Cada radio prorrateado entre varios circuitos queda con varias filas cuyo `peso_area` suma 1.0; `match_limpio=True` si el radio cayó entero dentro de un único circuito.

In [2]:
circuitos = cargar_circuitos_electorales(SOCIOECONOMIA / "circuitos_electorales_la_plata.geojson")
print(f"circuitos electorales de La Plata: {len(circuitos)}")

partes = []
for anio, nombre_archivo in [(2010, "radios_censales_2010_la_plata.geojson"), (2022, "radios_censales_2022_la_plata.geojson")]:
    radios = cargar_radios_censales(SOCIOECONOMIA / nombre_archivo, anio)
    print(f"radios censales {anio}: {len(radios)}")
    partes.append(calcular_correspondencia(circuitos, radios))

correspondencia = pd.concat(partes, ignore_index=True)
destino = SOCIOECONOMIA / "circuito_radio_correspondencia.csv"
correspondencia.to_csv(destino, index=False)
print(f"\n{len(correspondencia)} filas -> {destino}")

circuitos electorales de La Plata: 68
radios censales 2010: 849


radios censales 2022: 1049



3057 filas -> /workspaces/analisis-politica-economia/data/socioeconomia/circuito_radio_correspondencia.csv


In [3]:
# Cobertura: cuántos radios de cada censo tienen match limpio (un solo circuito) vs. prorrateado.
# Un radio prorrateado no es un error -- es real que los límites de circuito y de radio censal no
# coinciden -- pero cualquier cifra por circuito construida sobre esas filas es una estimación por
# área, no un conteo censal (ver README).
for anio in sorted(correspondencia["censo_anio"].unique()):
    sub = correspondencia[correspondencia["censo_anio"] == anio]
    n_radios = sub["radio_censal_id"].nunique()
    n_limpios = sub.loc[sub["match_limpio"], "radio_censal_id"].nunique()
    n_circuitos = sub["circuito_id"].nunique()
    print(f"{anio}: {n_radios} radios, {n_limpios} ({n_limpios/n_radios:.1%}) con match limpio, "
          f"{n_circuitos}/{len(circuitos)} circuitos con al menos un radio asociado")

circuitos_limite_incierto = {"493", "496F", "504C"}  # ya señalados en README (§ circuito_id canónico)
presentes = set(circuitos["circuito_id"])
print(f"\ncircuitos de límite incierto presentes en la capa de circuitos electorales: "
      f"{sorted(circuitos_limite_incierto & presentes)}")
print(f"ausentes de la capa (no se pudo bajar su polígono): {sorted(circuitos_limite_incierto - presentes)}")

2010: 849 radios, 454 (53.5%) con match limpio, 68/68 circuitos con al menos un radio asociado
2022: 1049 radios, 585 (55.8%) con match limpio, 68/68 circuitos con al menos un radio asociado

circuitos de límite incierto presentes en la capa de circuitos electorales: ['493', '496F']
ausentes de la capa (no se pudo bajar su polígono): ['504C']


## 2. EPH Gran La Plata — serie trimestral

`AGLOMERADO=2` (confirmado empíricamente: en la base individual del 1er trimestre de 2018 concentra 870.693 personas ponderadas, en línea con la población conocida de Gran La Plata). Cubre 2003T3-2025T4 (86 de 90 trimestres; faltan 2007T3, 2015T3, 2015T4 y 2016T1 -- INDEC no publicó, ver docstring de `eph_client.py`): 2003T3-2015T4 vía la rama histórica (`descargar_trimestre_historico`/`leer_base_historica`, microdatos DBF recuperados de Wayback Machine) y 2016T1-2025T4 con la rama regular de INDEC (`descargar_trimestre`/`leer_base`), que ya resuelve los nombres de archivo irregulares de 2016T2-T4 y 2017T1. **2003T3 es el piso real**: la EPH continua (el formato de panel rotativo que este cliente asume) reemplazó a la EPH puntual/"onda" a mediados de 2003 -- no hay captura de continua anterior a ese trimestre, y la "onda" es una encuesta de diseño distinto, no una extensión directa de esta serie. Genera tres CSV: `eph_gran_la_plata.csv` (una fila por trimestre), y `eph_gran_la_plata_por_sexo.csv`/`_por_edad.csv` (el núcleo laboral abierto por sexo y por tramo etario, una fila por (trimestre, categoría)).

**Nota de calidad del dato, 2007-2015**: este tramo coincide con la intervención del INDEC -- ampliamente documentada para el IPC, con dudas también planteadas sobre otras estadísticas oficiales del período. Las variables laborales core (`ESTADO`/`CAT_OCUP`, de donde salen `tasa_actividad`/`tasa_empleo`/`tasa_desocupacion`) no son el foco de esa controversia, pero las variables de ingreso (`P21`/`P47T`/`IPCF`, entonces `ingreso_*`/`ipcf_medio`) sí podrían llevar el mismo sesgo que otras series de precios/ingresos de la época -- no se corrige acá (no hay una fuente alternativa confiable a nivel Gran La Plata para ese período), solo se documenta como limitación conocida al leer esos años.

In [4]:
client = EphClient(cache_dir=SOCIOECONOMIA / "eph_cache")


def _trimestres_historicos():
    # _WAYBACK_DBF (eph_client.py) tiene capturas confirmadas 2003T3-2015T2 --
    # 2003T1/T2 no existen (la EPH continua arrancó en T3 2003, ver docstring
    # de eph_client.py) y quedan afuera solas via UrlDesconocida (no están en
    # el dict). 2015T3 en adelante (incl. 2015T4/2016T1, sin publicar) va por
    # la rama regular: 2016T2-T4 tienen nombre de archivo irregular pero
    # confirmado (_URLS_IRREGULARES), no son parte del histórico DBF/Wayback.
    for anio in range(2003, 2016):
        for trimestre in (1, 2, 3, 4):
            yield anio, trimestre


def _trimestres_regulares():
    for anio in range(2016, 2026):
        for trimestre in (1, 2, 3, 4):
            yield anio, trimestre


filas_gran_la_plata = []
filas_por_sexo = []
filas_por_edad = []

for anio, trimestre in _trimestres_historicos():
    try:
        archivo = client.descargar_trimestre_historico(anio, trimestre)
        individual = client.leer_base_historica(archivo, "individual")
        hogar = client.leer_base_historica(archivo, "hogar")
    except (TrimestreNoPublicado, UrlDesconocida) as e:
        print(f"{anio} T{trimestre}: {e}")
        continue
    filas_gran_la_plata.append(agregados_gran_la_plata(individual, hogar))
    filas_por_sexo.extend(agregados_por_sexo(individual))
    filas_por_edad.extend(agregados_por_edad(individual))

for anio, trimestre in _trimestres_regulares():
    try:
        zip_path = client.descargar_trimestre(anio, trimestre)
        individual = client.leer_base(zip_path, "individual")
        hogar = client.leer_base(zip_path, "hogar")
    except (TrimestreNoPublicado, UrlDesconocida) as e:
        print(f"{anio} T{trimestre}: {e}")
        continue
    filas_gran_la_plata.append(agregados_gran_la_plata(individual, hogar))
    filas_por_sexo.extend(agregados_por_sexo(individual))
    filas_por_edad.extend(agregados_por_edad(individual))

eph_gran_la_plata = pd.DataFrame(filas_gran_la_plata).sort_values(["anio", "trimestre"]).reset_index(drop=True)
destino_eph = SOCIOECONOMIA / "eph_gran_la_plata.csv"
eph_gran_la_plata.to_csv(destino_eph, index=False)
print(f"\n{len(eph_gran_la_plata)} trimestres -> {destino_eph}")

eph_por_sexo = pd.DataFrame(filas_por_sexo).sort_values(["anio", "trimestre", "sexo"]).reset_index(drop=True)
destino_sexo = SOCIOECONOMIA / "eph_gran_la_plata_por_sexo.csv"
eph_por_sexo.to_csv(destino_sexo, index=False)
print(f"{len(eph_por_sexo)} filas -> {destino_sexo}")

eph_por_edad = pd.DataFrame(filas_por_edad).sort_values(["anio", "trimestre", "tramo_etario"]).reset_index(drop=True)
destino_edad = SOCIOECONOMIA / "eph_gran_la_plata_por_edad.csv"
eph_por_edad.to_csv(destino_edad, index=False)
print(f"{len(eph_por_edad)} filas -> {destino_edad}")

eph_gran_la_plata.tail()

2003 T1: No hay una captura de Wayback Machine confirmada para 2003 T1 
2003 T2: No hay una captura de Wayback Machine confirmada para 2003 T2 


2007 T3: INDEC no publicó la EPH para 2007 T3.


2015 T3: INDEC no publicó la EPH para 2015 T3.
2015 T4: INDEC no publicó la EPH para 2015 T4.
2016 T1: INDEC no publicó la EPH para 2016 T1 (emergencia estadística o no relevamiento).



86 trimestres -> /workspaces/analisis-politica-economia/data/socioeconomia/eph_gran_la_plata.csv
172 filas -> /workspaces/analisis-politica-economia/data/socioeconomia/eph_gran_la_plata_por_sexo.csv
344 filas -> /workspaces/analisis-politica-economia/data/socioeconomia/eph_gran_la_plata_por_edad.csv


,anio,trimestre,tasa_actividad,tasa_empleo,tasa_desocupacion,tasa_informalidad,ingreso_ocupacion_principal_medio_todos_ocupados,ingreso_ocupacion_principal_medio_perceptores,pct_patron,pct_cuentapropia,...,pct_agua_red_publica,pct_vivienda_propia,pct_inquilino,tamanio_hogar_medio,pct_hogares_ayuda_social_gobierno,pct_hogares_prestamo_bancario,pct_hogares_vendio_pertenencias,ingreso_total_individual_medio_todos,ingreso_total_individual_medio_perceptores,ipcf_medio
81,2024,4,0.579023,0.532300,0.080693,0.358615,540576.145274,673858.471229,0.018224,0.182688,...,0.953664,0.649065,0.234460,2.560940,0.124404,0.156920,0.092576,439840.380789,563260.765454,602173.349588
82,2025,1,0.543540,0.496400,0.086728,0.421460,571094.805170,697898.598316,0.035899,0.160359,...,0.953056,0.697267,0.213231,2.637517,0.168716,0.132423,0.098650,450270.231606,592589.370038,704607.202897
83,2025,2,0.540857,0.503483,0.069100,0.343974,650117.687583,735699.709385,0.010898,0.173791,...,0.962423,0.694581,0.195012,2.756342,0.145196,0.108687,0.092318,487848.588534,579586.642474,643531.542057
84,2025,3,0.573842,0.527265,0.081167,0.355506,652148.090521,799604.060720,0.040975,0.197651,...,0.940022,0.654237,0.231253,2.595695,0.127258,0.122150,0.113582,553119.642804,711141.115895,757610.648866
85,2025,4,0.571559,0.517369,0.094811,0.321156,638455.698671,923270.635336,0.032544,0.249340,...,0.928638,0.668996,0.234618,2.672298,0.157950,0.217696,0.161890,473188.497020,695815.277736,725480.853385


## 3. Join con Censo por circuito (pendiente de la extracción manual REDATAM)

`censo_2010_radio.csv` / `censo_2022_radio.csv` no existen todavía en este repositorio — su extracción es manual (ver `data/socioeconomia/EXTRACCION_REDATAM.md`). Esta celda deja armado el join que hay que correr apenas existan: cada variable censal por radio se multiplica por `peso_area` antes de sumar por circuito, para que los radios prorrateados aporten solo la porción de su área que cae en ese circuito.

In [5]:
def unir_censo_a_circuitos(censo_radio: pd.DataFrame, censo_anio: int, columnas_variables: list[str]) -> pd.DataFrame:
    """censo_radio: una fila por radio_censal_id, con las columnas de columnas_variables ya numéricas."""
    corr = correspondencia[correspondencia["censo_anio"] == censo_anio]
    unido = corr.merge(censo_radio, on="radio_censal_id", how="left")
    for col in columnas_variables:
        unido[col] = unido[col] * unido["peso_area"]
    return unido.groupby("circuito_id")[columnas_variables].sum().reset_index()

for anio, nombre in [(2010, "censo_2010_radio.csv"), (2022, "censo_2022_radio.csv")]:
    ruta = SOCIOECONOMIA / nombre
    if ruta.exists():
        censo_radio = pd.read_csv(ruta, dtype={"radio_censal_id": str})
        columnas_variables = [c for c in censo_radio.columns if c != "radio_censal_id"]
        censo_por_circuito = unir_censo_a_circuitos(censo_radio, anio, columnas_variables)
        destino = SOCIOECONOMIA / f"censo_{anio}_circuito.csv"
        censo_por_circuito.to_csv(destino, index=False)
        print(f"censo {anio}: {len(censo_por_circuito)} circuitos -> {destino}")
    else:
        print(f"censo {anio}: falta {ruta.name} (extracción manual pendiente, ver EXTRACCION_REDATAM.md)")

censo 2010: falta censo_2010_radio.csv (extracción manual pendiente, ver EXTRACCION_REDATAM.md)
censo 2022: falta censo_2022_radio.csv (extracción manual pendiente, ver EXTRACCION_REDATAM.md)


## Limitaciones 

1. **EPH = aglomerado Gran La Plata (La Plata+Berisso+Ensenada), nunca por circuito.** Cualquier análisis que cruce la serie EPH con resultados electorales por circuito está mezclando grados de agregación distintos — señalarlo explícitamente, no forzarlo a un único denominador.
2. **Censo 2022 describe la estructura cerca de 2022**, no las elecciones tempranas de este proyecto (2011-2015) — no usarlo para explicarlas sin decirlo.
3. **Los radios prorrateados no son un caso raro**: bastante más de un tercio de los radios de La Plata (ver cobertura arriba) cruzan el límite de más de un circuito. Cualquier cifra censal por circuito construida a partir de esas filas es una estimación por área, no un conteo censal.
4. **Comparar Censo 2010 vs. 2022 exige pasar por la geometría, no por el `radio_censal_id` crudo** — los radios cambian de límites entre censos (ver `EXTRACCION_REDATAM.md`).